In [ ]:
# BLOCK 1 – Setup imports, paths, flags, and constants

import gc
import traceback
from pathlib import Path
import cv2
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from totalsegmentator.python_api import totalsegmentator
from ultralytics import YOLO
from tqdm import tqdm
import shutil, time
import matplotlib.pyplot as plt


# ── PATHS (EDIT HERE) ────────────────────────────────────────
MODEL_PATH = Path(r"C:\Users\Ryan Krishna\Documents\Overscanning\YOLO\model_and_training\yolo11_pubic_symphysis_m_hardtrain\weights\best.pt")
NIFTI_DIR = Path(r"C:\Users\Ryan Krishna\Documents\tests")
CSV_PATH = NIFTI_DIR / "overscanning_results.csv"

# ── FLAGS ─────────────────────────────────────────────────────
DISPLAY_DETECTION = True
FAST_MODEL = False
MULTI_LABEL_MASK = True

# ── CONSTANTS ─────────────────────────────────────────────────
FINAL_CONF = 0.20
BACKGROUND_HU = -300
model = YOLO(str(MODEL_PATH))

In [ ]:
# BLOCK 2 – Pubic symphysis detection → caudal overscan (femur-aware)

import contextlib, io

if CSV_PATH.exists():
    done_df = pd.read_csv(CSV_PATH)
    done_set = set(done_df["file_name"].tolist())
else:
    done_set = set()


def preprocess_slice(arr: np.ndarray) -> np.ndarray:
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    arr = (arr * 255).astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)


def ensure_femur_mask(ct_path: Path) -> Path | None:
    out_dir     = ct_path.parent / "ts_femur"
    fem_l_path  = out_dir / "femur_left.nii.gz"
    fem_r_path  = out_dir / "femur_right.nii.gz"
    merged_path = ct_path.parent / "femur_combined.nii.gz"

    if merged_path.exists():
        return merged_path

    if not (fem_l_path.exists() and fem_r_path.exists()):
        out_dir.mkdir(exist_ok=True)
        for dev in ("gpu", "cpu"):
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    totalsegmentator(
                        ct_path, out_dir,
                        roi_subset=["femur_left", "femur_right"],
                        task="total",
                        fast=FAST_MODEL,
                        device=dev,
                    )
                break
            except Exception as e:
                print(f"TS({dev}) {ct_path.name}: {e}")
        else:
            return None

    try:
        fem_l = nib.load(fem_l_path).get_fdata() > 0
        fem_r = nib.load(fem_r_path).get_fdata() > 0
    except FileNotFoundError:
        return None

    merged = (fem_l | fem_r).astype(np.uint8)
    if not merged.any():
        return None

    ref = nib.load(fem_l_path if fem_l_path.exists() else fem_r_path)
    nib.save(nib.Nifti1Image(merged, ref.affine, ref.header), merged_path)

    for p in (fem_l_path, fem_r_path):
        if p.exists():
            p.unlink()
    if out_dir.exists() and not any(out_dir.iterdir()):
        out_dir.rmdir()
    return merged_path


def femur_top_info(ct_path: Path) -> tuple[int, float] | None:
    m = ensure_femur_mask(ct_path)
    if m is None:
        return None
    mask = nib.load(str(m))
    mask_np = mask.get_fdata() > 0
    slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if slices.size == 0:
        return None
    affine = mask.affine
    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in slices]
    return max(z_coords, key=lambda t: t[1])

DEVICE = 0 if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu"

def find_valid_pubic_slice(ct_path: Path, z_cutoff_mm: float) -> int | None:
    ct = nib.load(str(ct_path))
    affine = ct.affine
    vol = ct.get_fdata()
    H, W, Z = vol.shape

    best_conf, best_slice = -1.0, None
    for z in range(Z):
        if float((affine @ [0, 0, z, 1])[2]) > z_cutoff_mm:
            continue
        img = preprocess_slice(vol[:, :, z])
        res = model.predict(img, conf=FINAL_CONF, device=DEVICE, save=False, verbose=False)[0]
        for b in sorted(res.boxes, key=lambda bb: float(bb.conf), reverse=True):
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            if vol[int((y1+y2)/2), int((x1+x2)/2), z] <= BACKGROUND_HU:
                continue
            cx = int((x1+x2)/2)
            if abs(cx - W // 2) > 0.20 * W:
                continue
            win = vol[max(0, int((y1+y2)/2) - 10):min(H, int((y1+y2)/2) + 10),
                      max(0, cx - 10):min(W, cx + 10), z]
            if win.mean() < 150:
                continue
            conf = float(b.conf)
            if conf > best_conf:
                best_conf, best_slice = conf, z
            break
    return best_slice


def is_ct_vol(p: Path) -> bool:
    if p.parent.name.startswith("ts_"):
        return False
    if p.name.endswith("_combined.nii.gz"):
        return False
    if p.name.startswith(("femur_", "liver_", "spleen_")):
        return False
    return True


nii_paths = [p for p in NIFTI_DIR.rglob("*.nii*") if is_ct_vol(p)]
print(f"{len(nii_paths)} CT volumes found")


for ct_path in tqdm(nii_paths, desc="Processing", unit="vol"):
    if ct_path.name in done_set:
        continue
    try:
        fem_data = femur_top_info(ct_path)
        if fem_data:
            fem_slice, fem_top_z = fem_data
            z_cut = fem_top_z
        else:
            fem_slice, fem_top_z = None, np.nan
            z_cut = float("inf")

        pubic_slice = find_valid_pubic_slice(ct_path, z_cut)
        if pubic_slice is None and fem_slice is not None:
            pubic_slice, source_label = fem_slice, "FemurFallback"
        elif pubic_slice is None:
            continue
        else:
            source_label = "YOLO" if not np.isnan(fem_top_z) else "YOLO_NoFemur"

        ct_img  = nib.load(str(ct_path))
        affine  = ct_img.affine
        Z       = ct_img.shape[2]
        pubic_z = float((affine @ [0, 0, pubic_slice, 1])[2])
        end_z   = min(float((affine @ [0, 0, k, 1])[2]) for k in range(Z))
        caudal  = abs(end_z - pubic_z)

        row = {
            "file_name": ct_path.name,
            "pubic_z_mm": int(round(pubic_z)),
            "scan_end_z_mm": int(round(end_z)),
            "caudal_overscan_mm": int(round(caudal)),
            "femur_top_z_mm": (int(round(fem_top_z))
                               if not np.isnan(fem_top_z) else np.nan),
            "pubic_source": source_label,
        }

        if CSV_PATH.exists():
            df = pd.read_csv(CSV_PATH)
            if row["file_name"] in df["file_name"].values:
                ix = df.index[df["file_name"] == row["file_name"]][0]
                for col in row:
                    df.at[ix, col] = row[col]
            else:
                df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
            df.sort_values("file_name").to_csv(CSV_PATH, index=False)
        else:
            pd.DataFrame([row]).to_csv(CSV_PATH, index=False)

        done_set.add(ct_path.name)

    except Exception as e:
        traceback.print_exc(limit=1)

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"Finished. CSV now contains {len(done_set)} rows")

In [ ]:
# BLOCK 3 – Liver & spleen segmentation

def ensure_liver_spleen_mask(ct_path: Path) -> Path | None:
    """Return merged liver–spleen mask, creating it if missing."""
    out_dir      = ct_path.parent / "ts_liver_spleen"
    liver_mask   = out_dir / "liver.nii.gz"
    spleen_mask  = out_dir / "spleen.nii.gz"
    merged_path  = ct_path.parent / "liver_spleen_combined.nii.gz"

    if merged_path.exists():
        return merged_path

    if not (liver_mask.exists() and spleen_mask.exists()):
        out_dir.mkdir(exist_ok=True)
        for dev in ("gpu", "cpu"):
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    totalsegmentator(
                        ct_path,
                        out_dir,
                        roi_subset=["liver", "spleen"],
                        task="total",
                        fast=FAST_MODEL,
                        device=dev,
                    )
                break
            except Exception as e:
                print(f"TS({dev}) {ct_path.name}: {e}")
        else:
            return None

    try:
        liver_data  = nib.load(liver_mask ).get_fdata() > 0
        spleen_data = nib.load(spleen_mask).get_fdata() > 0
    except FileNotFoundError:
        return None

    if MULTI_LABEL_MASK:
        combined = np.zeros(liver_data.shape, np.uint8)
        combined[liver_data]  = 1
        combined[spleen_data] = 2
    else:
        combined = (liver_data | spleen_data).astype(np.uint8)

    ref_img = nib.load(liver_mask if liver_mask.exists() else spleen_mask)
    nib.save(nib.Nifti1Image(combined, ref_img.affine, ref_img.header), merged_path)

    for p in (liver_mask, spleen_mask):
        if p.exists():
            p.unlink()
    if out_dir.exists() and not any(out_dir.iterdir()):
        out_dir.rmdir()

    return merged_path


patients = sorted([p for p in NIFTI_DIR.iterdir() if p.is_dir()])
print(f"{len(patients)} patient folders found")

for pat in tqdm(patients, desc="Processing", unit="pat"):
    try:
        merged_path = pat / "liver_spleen_combined.nii.gz"
        if merged_path.exists():
            continue

        ct_candidates = [
            f for f in pat.glob("*.nii*")
            if f.name.startswith(pat.name)
            and "femur_combined" not in f.name.lower()
            and "liver_spleen_combined" not in f.name.lower()
            and not f.name.startswith("ts_")
        ]

        if not ct_candidates:
            continue

        ensure_liver_spleen_mask(ct_candidates[0])

    except Exception as e:
        traceback.print_exc(limit=1)

print("Finished. All folders now have a combined liver & spleen segmentation.")

In [ ]:
# BLOCK 4 – Cranial overscan

def is_ct_vol(p: Path) -> bool:
    """True for original CT volumes, False for masks/segmentation outputs."""
    if p.parent.name.startswith("ts_"):
        return False
    if p.name.startswith("ts_"):
        return False
    if p.name.endswith("_combined.nii.gz"):
        return False
    if p.name.startswith(("femur_", "liver_", "spleen_")):
        return False
    return True


ct_paths = [p for p in NIFTI_DIR.rglob("*.nii*") if is_ct_vol(p)]
print(f"{len(ct_paths)} CT volumes found")

if CSV_PATH.exists():
    csv_df = pd.read_csv(CSV_PATH)
else:
    csv_df = pd.DataFrame(columns=[
        "file_name", "pubic_z_mm", "scan_end_z_mm", "caudal_overscan_mm",
        "femur_top_z_mm", "pubic_source",
        "liver_spleen_z_mm", "scan_start_z_mm", "cranial_overscan_mm", "top_organ"
    ])


def cranial_overscan(ct_path: Path, mask_path: Path) -> tuple[int, int, int, str]:
    """Compute cranial overscan and identify the top organ (liver/spleen)."""
    ct_img   = nib.load(str(ct_path))
    mask_img = nib.load(str(mask_path))
    affine   = ct_img.affine
    mask_np  = mask_img.get_fdata()

    seg_slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if seg_slices.size == 0:
        raise RuntimeError("combined mask empty")

    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in seg_slices]

    Z = ct_img.shape[2]
    z_edge0 = float((affine @ [0, 0,      0, 1])[2])
    z_edgeN = float((affine @ [0, 0, Z - 1, 1])[2])
    cranial_edge_z = max(z_edge0, z_edgeN)

    highest_slice, highest_z = min(z_coords, key=lambda t: abs(t[1] - cranial_edge_z))

    labels     = mask_np[:, :, highest_slice][mask_np[:, :, highest_slice] > 0].astype(int)
    organ_map  = {1: "Liver", 2: "Spleen"}
    organ_top  = organ_map.get(int(np.bincount(labels).argmax()), "Unknown")

    cranial_mm    = int(round(abs(cranial_edge_z - highest_z)))
    scan_start_mm = int(round(cranial_edge_z))
    organ_z_mm    = int(round(highest_z))
    return cranial_mm, organ_z_mm, scan_start_mm, organ_top


for ct_path in tqdm(ct_paths, desc="Processing", unit="vol"):
    mask_path = ct_path.parent / "liver_spleen_combined.nii.gz"
    if not mask_path.exists():
        continue

    try:
        cranial_mm, organ_z_mm, scan_start_mm, organ_top = cranial_overscan(ct_path, mask_path)
    except Exception:
        traceback.print_exc(limit=1)
        continue

    row = {
        "file_name"          : ct_path.name,
        "liver_spleen_z_mm"  : organ_z_mm,
        "scan_start_z_mm"    : scan_start_mm,
        "cranial_overscan_mm": cranial_mm,
        "top_organ"          : organ_top,
    }

    if ct_path.name in csv_df["file_name"].values:
        ix = csv_df.index[csv_df["file_name"] == ct_path.name][0]
        for k, v in row.items():
            csv_df.at[ix, k] = v
    else:
        csv_df = pd.concat([csv_df, pd.DataFrame([row])], ignore_index=True)

    csv_df.sort_values("file_name").to_csv(CSV_PATH, index=False)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Finished. Cranial metrics calculated and appended to CSV.")

In [ ]:
# BLOCK 5 - Generate preview MP4s of mid-coronal slices

OUT_DIR = NIFTI_DIR.parent / "trauma_overscan_videos_test"
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
req_cols = {"file_name", "pubic_source", "pubic_z_mm", "liver_spleen_z_mm", "top_organ"}
if not req_cols.issubset(df.columns):
    raise KeyError("CSV missing required columns")

print(f"{len(df)} CT volumes found")


def build_mp4(
    scan_id: str,
    pubic_z_mm: float,
    organ_z_mm: float,
    organ_label: str,
    pubic_source: str,
    fps: int = 48,
    slice_span: int = 100,
):
    """Create {scan_id}.mp4 preview video."""
    folder = NIFTI_DIR / scan_id
    if not folder.is_dir():
        raise FileNotFoundError(f"{folder} not found")

    ct_candidates = [
        f for f in folder.glob("*.nii*")
        if f.name.startswith(scan_id)
        and "femur_combined" not in f.name.lower()
        and "liver_spleen_combined" not in f.name.lower()
        and not f.name.startswith("ts_")
    ]
    if not ct_candidates:
        raise FileNotFoundError("CT volume not found")
    if len(ct_candidates) > 1:
        raise RuntimeError(f"Multiple CT volumes: {[c.name for c in ct_candidates]}")
    ct_path = ct_candidates[0]

    fem_path = ensure_femur_mask(ct_path)
    if fem_path is None or not fem_path.exists():
        raise FileNotFoundError("Femur mask missing")

    org_path = ensure_liver_spleen_mask(ct_path)
    if org_path is None or not org_path.exists():
        raise FileNotFoundError("Liver-spleen mask missing")

    mp4_path = OUT_DIR / f"{scan_id}.mp4"
    if mp4_path.exists():
        mp4_path.unlink()

    ct_img  = nib.load(str(ct_path))
    vol     = ct_img.get_fdata()
    fem_msk = nib.load(str(fem_path)).get_fdata() > 0
    org_msk = nib.load(str(org_path)).get_fdata() > 0
    affine  = ct_img.affine
    vx, _, vz = ct_img.header.get_zooms()[:3]

    _, Y, Z = vol.shape
    z_world = np.flip((affine @ np.vstack([np.zeros(Z), np.zeros(Z), np.arange(Z), np.ones(Z)]))[2])
    pubic_row = int(np.argmin(np.abs(z_world - pubic_z_mm)))
    organ_row = int(np.argmin(np.abs(z_world - organ_z_mm)))

    mid_y, half = Y // 2, slice_span // 2
    start_y, end_y = max(0, mid_y - half), min(Y - 1, mid_y + half)
    y_stretch = vz / vx

    font, fs, th = cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2
    red, green = (0, 0, 255), (0, 255, 0)
    landmark = "Femur" if pubic_source == "FemurFallback" else "Pubic Symphysis"

    def render(y_idx: int):
        ct = np.flipud(vol[:, y_idx, :].T)
        fm = np.flipud(fem_msk[:, y_idx, :].T)
        om = np.flipud(org_msk[:, y_idx, :].T)

        img = np.clip((ct + 200) / 500, 0, 1) * 255
        img = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_GRAY2BGR)

        overlay = np.zeros_like(img)
        overlay[fm] = (255, 0, 0)
        overlay[om] = (0, 255, 255)
        img = cv2.addWeighted(img, 0.8, overlay, 0.25, 0)

        if y_stretch != 1.0:
            h, w = img.shape[:2]
            img = cv2.resize(img, (w, int(h * y_stretch)), interpolation=cv2.INTER_CUBIC)

        h, w = img.shape[:2]
        cv2.line(img, (0, int(pubic_row * y_stretch)), (w - 1, int(pubic_row * y_stretch)), red, 2)
        cv2.line(img, (0, int(organ_row * y_stretch)), (w - 1, int(organ_row * y_stretch)), green, 2)

        cv2.putText(img, f"{landmark} z={pubic_z_mm:.0f} mm",
                    (10, max(20, int(pubic_row * y_stretch) - 6)), font, fs, red, th, cv2.LINE_AA)
        cv2.putText(img, f"{organ_label} z={organ_z_mm:.0f} mm",
                    (10, min(h - 10, int(organ_row * y_stretch) + 20)), font, fs, green, th, cv2.LINE_AA)
        cv2.putText(img, f"{scan_id} | y={y_idx}",
                    (10, h - 10), font, fs, (255, 255, 0), th, cv2.LINE_AA)
        return img

    first = render(start_y)
    h, w = first.shape[:2]
    vw = cv2.VideoWriter(str(mp4_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    vw.write(first)
    for y in range(start_y + 1, end_y + 1):
        vw.write(render(y))
    vw.release()


ok = failed = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing", unit="scan"):
    sid = row["file_name"].split(".nii")[0]
    try:
        build_mp4(
            sid,
            float(row["pubic_z_mm"]),
            float(row["liver_spleen_z_mm"]),
            str(row["top_organ"]).strip(),
            str(row["pubic_source"]).strip(),
        )
        ok += 1
    except Exception as e:
        failed += 1
        tqdm.write(f"✗ {sid}: {e}")
        traceback.print_exc()

print(f"Finished. {ok} MP4s created and saved to {OUT_DIR}")

In [ ]:
# BLOCK 6 – Overscan metrics & summary stats

SUMMARY_CSV_PATH = CSV_PATH.with_name("summary_statistics.csv")

df = pd.read_csv(CSV_PATH, sep=None, engine="python", encoding="utf-8-sig")
df.columns = df.columns.str.strip().str.replace("\ufeff", "", regex=False)

caudal_thresh = df["pubic_source"].eq("FemurFallback").map({True: 50, False: 30})
df["caudal_overscan?"]  = (df["caudal_overscan_mm"]  > caudal_thresh).map({True: "yes", False: "no"})
df["cranial_overscan?"] = df["cranial_overscan_mm"].gt(30).map({True: "yes", False: "no"})
df["overscanning?"]     = ((df["caudal_overscan?"] == "yes") | (df["cranial_overscan?"] == "yes")).map({True: "yes", False: "no"})

df["calc_caudal_overscan_mm"]  = (df["caudal_overscan_mm"]  - caudal_thresh).clip(lower=0)
df["calc_cranial_overscan_mm"] = (df["cranial_overscan_mm"] - 30).clip(lower=0)

df["calc_total_overscan_mm"] = df["calc_caudal_overscan_mm"] + df["calc_cranial_overscan_mm"]
scan_length = (df["scan_end_z_mm"] - df["scan_start_z_mm"]).replace(0, pd.NA)
percent_vals = (df["calc_total_overscan_mm"] / scan_length * 100).abs().round()
df["%_overscan"] = percent_vals.apply(lambda x: f"{int(x)}%" if pd.notna(x) else pd.NA)

sep = "\t" if "\t" in open(CSV_PATH, encoding="utf-8-sig").readline() else ","
df.to_csv(CSV_PATH, index=False, sep=sep)
print("Main CSV updated.")

caudal_excess  = df["calc_caudal_overscan_mm"]
cranial_excess = df["calc_cranial_overscan_mm"]
total_excess   = df["calc_total_overscan_mm"]

m_caudal = caudal_excess [caudal_excess  > 0].mean()
s_caudal = caudal_excess [caudal_excess  > 0].std()
m_cran   = cranial_excess[cranial_excess > 0].mean()
s_cran   = cranial_excess[cranial_excess > 0].std()
m_total  = total_excess  [total_excess   > 0].mean()
s_total  = total_excess  [total_excess   > 0].std()

def fmt_mm(val):  return f"{int(round(val))} mm" if pd.notna(val) else "-"
def fmt_pct(val): return f"{int(round(val))}%"  if pd.notna(val) else "-"

summary = pd.DataFrame({
    "METRIC": [
        "n",
        "Mean_caudal_overscan_excess",
        "SD_caudal_overscan_excess",
        "Mean_cranial_overscan_excess",
        "SD_cranial_overscan_excess",
        "Mean_total_overscan_excess",
        "SD_total_overscan_excess",
        "%_caudal_overscan",
        "%_cranial_overscan",
        "%_overscanning",
    ],
    "VALUE": [
        len(df),
        fmt_mm(m_caudal),
        fmt_mm(s_caudal),
        fmt_mm(m_cran),
        fmt_mm(s_cran),
        fmt_mm(m_total),
        fmt_mm(s_total),
        fmt_pct((df['caudal_overscan?']=='yes').mean()*100),
        fmt_pct((df['cranial_overscan?']=='yes').mean()*100),
        fmt_pct((df['overscanning?']=='yes').mean()*100),
    ],
})

summary.to_csv(SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Summary statistics written → {SUMMARY_CSV_PATH}")

In [ ]:
# BLOCK 7 – Save figures
def save_fig(
    fig,
    name: str,
    out_dir: Path = NIFTI_DIR,
    exts: tuple[str, ...] = ("png",),
    dpi: int = 300,
    close: bool = True,
) -> None:
    """Save *fig* to <out_dir>/<name>.<ext> for each extension in *exts*."""
    out_dir.mkdir(parents=True, exist_ok=True)
    for ext in exts:
        fig.savefig(out_dir / f"{name}.{ext}", dpi=dpi, bbox_inches="tight")
    if close:
        plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4.5))
x_idx   = np.arange(len(df))
cranial = df["calc_cranial_overscan_mm"].values
caudal  = df["calc_caudal_overscan_mm"].values

ax.scatter(x_idx,  cranial, s=10, label="Cranial")
ax.scatter(x_idx, -caudal,  s=10, label="Caudal")

max_abs = np.nanmax([np.abs(cranial).max(), np.abs(caudal).max()])
ax.set_ylim(-max_abs * 1.05, max_abs * 1.05)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Case index")
ax.set_ylabel("Overscan excess (mm)\n(+ cranial / − caudal)")
ax.set_title("Cranial vs Caudal Overscan Excess")
ax.legend(frameon=False)
plt.tight_layout()
save_fig(fig, "scatter_cranial_caudal")
print(f"Scatterplot saved to {NIFTI_DIR / 'scatter_cranial_caudal.png'}")

box_cols   = ["calc_cranial_overscan_mm",
              "calc_caudal_overscan_mm",
              "calc_total_overscan_mm"]
box_labels = ["Cranial", "Caudal", "Total"]
data       = [df[c].dropna().values for c in box_cols]

fig, ax = plt.subplots(figsize=(5, 7))
bp = ax.boxplot(
    data, vert=True, whis=1.5, showmeans=True, meanline=True,
    showcaps=True, showfliers=True, widths=0.6, patch_artist=True,
    medianprops=dict(color="black", linewidth=1.5),
    meanprops=dict(color="black", linestyle="--", linewidth=1),
    whiskerprops=dict(color="black", linestyle="--", linewidth=1),
    capprops=dict(color="black", linewidth=1),
    flierprops=dict(marker="o", markersize=4, markerfacecolor="none",
                    markeredgecolor="black", alpha=0.8),
)
for patch in bp["boxes"]:
    patch.set_facecolor("#1f77b4")
    patch.set_edgecolor("black")

ax.set_xticks(range(1, len(box_labels) + 1))
ax.set_xticklabels(box_labels)
ax.set_ylabel("Overscan excess (mm)")
ax.set_title("Overscan Excess – Box & Whisker")
plt.tight_layout()
save_fig(fig, "box_cranial_caudal_total")
print(f"Box-and-whisker plot saved to {NIFTI_DIR / 'box_cranial_caudal_total.png'}")

fig, ax = plt.subplots(figsize=(6, 4.5))
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ax.bar(x_idx,  cranial, width=0.8, color=colors[0], label="Cranial")
ax.bar(x_idx, -caudal,  width=0.8, color=colors[1], label="Caudal")

max_abs = np.nanmax([np.abs(cranial).max(), np.abs(caudal).max()])
ax.set_ylim(-max_abs * 1.05, max_abs * 1.05)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Case index")
ax.set_ylabel("Overscan excess (mm)\n(+ cranial / − caudal)")
ax.set_title("Cranial vs Caudal Overscan Excess – Bar Plot")
ax.legend(frameon=False)
plt.tight_layout()
save_fig(fig, "bar_cranial_caudal")
print(f"Bar plot saved to {NIFTI_DIR / 'bar_cranial_caudal.png'}")